# Gesture Drive AI


## Install dependencies


In [ ]:
%pip install pyserial mediapipe opencv-python


## Import libraries


In [ ]:
import time

import cv2
import mediapipe as mp
import serial


## Configuration


In [ ]:
ARDUINO_PORT = "COM12"
BAUD_RATE = 9600
CAMERA_INDEX = 0

STOP_COMMAND = "STOP"
FORWARD_COMMAND = "FORWARD"
LEFT_COMMAND = "LEFT"
RIGHT_COMMAND = "RIGHT"

previous_command = ""


## Gesture helpers


In [ ]:
def count_open_fingers(landmarks):
    fingers_open = 0

    if landmarks[8].y < landmarks[6].y:
        fingers_open += 1
    if landmarks[12].y < landmarks[10].y:
        fingers_open += 1
    if landmarks[16].y < landmarks[14].y:
        fingers_open += 1
    if landmarks[20].y < landmarks[18].y:
        fingers_open += 1

    return fingers_open


def classify_hand(landmarks):
    if count_open_fingers(landmarks) >= 3:
        return "OPEN"
    return "CLOSED"


def read_hand_states(results):
    hand_states = {"Left": "OFF_SCREEN", "Right": "OFF_SCREEN"}

    if not results.multi_hand_landmarks or not results.multi_handedness:
        return hand_states

    for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
        label = handedness.classification[0].label
        hand_states[label] = classify_hand(hand_landmarks.landmark)

    return hand_states


def choose_drive_command(hand_states):
    left_state = hand_states["Left"]
    right_state = hand_states["Right"]

    if left_state == "OFF_SCREEN" and right_state == "OFF_SCREEN":
        return STOP_COMMAND
    if right_state == "CLOSED" and left_state != "CLOSED":
        return RIGHT_COMMAND
    if left_state == "CLOSED" and right_state != "CLOSED":
        return LEFT_COMMAND
    if left_state == "OPEN" and right_state == "OPEN":
        return FORWARD_COMMAND
    return STOP_COMMAND


## Connect to Arduino and camera


In [ ]:
arduino = serial.Serial(ARDUINO_PORT, BAUD_RATE)
time.sleep(2)

mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=2)
draw = mp.solutions.drawing_utils
cap = cv2.VideoCapture(CAMERA_INDEX)


## Run gesture detection


In [ ]:
try:
    while True:
        ok, frame = cap.read()
        if not ok:
            continue

        frame = cv2.flip(frame, 1)
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        results = hands.process(rgb)
        hand_states = read_hand_states(results)
        command = choose_drive_command(hand_states)

        if results.multi_hand_landmarks:
            for hand in results.multi_hand_landmarks:
                draw.draw_landmarks(frame, hand, mp_hands.HAND_CONNECTIONS)

        if command != previous_command:
            arduino.write((command + "\n").encode())
            previous_command = command
            print("Sent:", command, hand_states)

        status_text = f"L:{hand_states['Left']} R:{hand_states['Right']} -> {command}"
        cv2.putText(frame, status_text, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow("Hand Detection", frame)

        if cv2.waitKey(1) == ord("q"):
            break
finally:
    cap.release()
    cv2.destroyAllWindows()
    arduino.close()


## Arduino serial commands

The notebook sends `FORWARD`, `LEFT`, `RIGHT`, and `STOP` over serial. Your Arduino sketch should map those commands to the four continuous-rotation servos.


## Manual cleanup


In [ ]:
if "cap" in globals():
    cap.release()

cv2.destroyAllWindows()

if "arduino" in globals() and arduino.is_open:
    arduino.close()
